# Double Source-Plane Lens — Direct Fit

## Learning to Autolens / Examples / double_source_plane

---

**Problem.** *One* lens galaxy at $z_L = 0.5$ deflects **two sources at different redshifts** ($z_{S1}=1.0$, $z_{S2}=2.5$). The two sources form distinct arc systems at different angular scales because their Einstein-effective radii depend on the angular-diameter-distance ratio $D_{ds}/D_s$, which differs between the two planes.

The ratio of observed Einstein radii is the cosmology-sensitive quantity

$$
\beta \;=\; \frac{D_{ds_1}/D_{s_1}}{D_{ds_2}/D_{s_2}} \;\approx\; 0.59 \quad \text{(FlatLambdaCDM, } H_0=70, \Omega_m=0.3\text{)}.
$$

DSPL systems are one of the few observational probes that constrain dark-energy parameters **without time delays** (Collett & Auger 2014).

**Architecture.** This is *not* a compound lens (see [`../compound_lens/`](../compound_lens/) for that). There's only ONE deflector; the complexity comes from **two source planes** sharing a common lens model.

**Method.** Single free-fit with `af.Nautilus`. `al.Tracer` handles multi-plane ray-tracing automatically when given galaxies at three different redshifts.

**Prerequisites.** Mod 03 (Nautilus free-fit), Mod 04 (SLaM), Examples/compound_lens (multi-plane Tracer).


In [ ]:
import os, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, IFrame, Markdown, display

import autofit as af
import autolens as al
import autolens.plot as aplt

%matplotlib inline
print(f"PyAutoLens version: {al.__version__}")

In [ ]:
RESULTS_ROOT = Path("results")

def show_result(stage_name):
    stage_dir = RESULTS_ROOT / stage_name
    if not stage_dir.exists():
        print(f"(no results for stage {stage_name!r} — run on Cannon or set LTA_RUN_HEAVY=1)")
        return
    sp = stage_dir / "summary.json"
    if sp.exists():
        s = json.loads(sp.read_text())
        display(Markdown(
            f"### `{stage_name}`\n"
            f"- log evidence: **{s.get('log_evidence'):.2f}**\n"
            f"- χ²/N = **{s.get('chi_squared_per_pixel'):.3f}**\n"
            f"- max |normalized residual| = **{s.get('max_abs_normalized_residual'):.2f} σ**"
        ))
    fp = stage_dir / "fit_subplot.png"
    if fp.exists():
        display(Markdown("**Fit subplot:**"))
        display(Image(filename=str(fp)))

---
## 1. Load the data

The mock is a 120×120 HST-like cutout (pixel_scales=0.05″) with two source-plane arc systems. Running `mocks/generate_mock.py` regenerates it from scratch.

In [ ]:
dataset = al.Imaging.from_fits(
    data_path      = Path("mocks") / "mock_image.fits",
    noise_map_path = Path("mocks") / "mock_noise.fits",
    psf_path       = Path("mocks") / "mock_psf.fits",
    pixel_scales   = 0.05,
)
mask = al.Mask2D.circular(
    shape_native = dataset.shape_native,
    pixel_scales = dataset.pixel_scales,
    radius       = 2.8,
)
dataset = dataset.apply_mask(mask=mask)
aplt.subplot_imaging_dataset(dataset=dataset)

---
## 2. Model composition (native multi-plane, one lens + two sources)

Three galaxies at three redshifts. `al.Tracer` groups them into planes automatically. The key pedagogical point: **the same `lens.mass` posterior must fit both source-plane arcs simultaneously** — if it doesn't, the fit is inconsistent with a single lens, which would tell us the system isn't DSPL.

The driver in `Modules/10_Cluster_Computing/scripts/fit_example_double_source_plane.py` contains the full prior setup (26 free parameters: 12 for the lens galaxy, 7 per source). Loose, minimally-informative priors — same philosophy as the compound_lens v3 fit. The priors:

- Lens mass: `einstein_radius ~ UniformPrior(0.5, 3.0)`, centre `GaussianPrior(0, 0.1)`, ell_comps `TruncatedGaussianPrior(0, 0.3, [-1,1])`, shear `γ ~ GaussianPrior(0, 0.1)`.
- Lens bulge: wide `LogUniform` intensity, `UniformPrior(0.1, 3)` effective_radius, `UniformPrior(0.8, 5)` sersic_index.
- Each source: SersicCore, centre `GaussianPrior(0, 0.3)`, compact `UniformPrior(0.02, 0.5)` effective_radius.

Read the driver for the authoritative prior list.

---
## 3. Results (loaded from Cannon)

Heavy Nautilus fit. Submitted via:

```bash
sbatch --export=ALL,EXAMPLE=double_source_plane,FIT_EXTRA_ARGS=--part=direct \
    Modules/10_Cluster_Computing/scripts/submit_cannon.slurm
```

When the fit lands, `export_results.py` writes `results/dspl_direct_fit/` with the canonical 7 files (fit_subplot.png, summary.json, model_results.txt, samples.csv, corner.pdf, info.txt, samples_summary.json). The cell below displays them:

In [ ]:
show_result("dspl_direct_fit")

---
## 4. Audit template

> ```
> Verdict:  [ PASS | SUSPECT | FAIL ]
>
> Numeric summary
>   chi²/N:           __.___          (pass: ≤1.3)
>   max |res|:        ___.__ σ         (pass: ≤4σ)
>   log_Z:            ___,___
>   free params:      26
>
> Panel walk
>   1. Residual map:          ___
>   2. Residual map 1σ:       ___
>   3. Chi² map:              ___ (hot spots on either arc?)
>   4. Source Plane (Zoomed): ___
>   5. Lens Light Subtracted: ___
>   6. model_results.txt:     ___ (θ_E₁/θ_E₂ ratio close to truth β?)
> ```

### What DSPL-specific things to look for

- **Einstein-effective radii ratio.** In the model_results.txt, the single fitted `lens.mass.einstein_radius` is defined for the reference source plane (autolens chooses one by convention). The *effective* Einstein radii for each source at their respective redshift are recovered from ray-tracing. Their ratio should match β ≈ 0.59.
- **Both sources in the source-plane panel.** The Source Plane (Zoomed) panel shows the reconstructed sources back in the source plane. Both should be compact galaxies.
- **Coherent arcs in the residual map are much worse in a DSPL than a single-source fit** — if the one mass model doesn't satisfy both source planes simultaneously, residuals appear as *distinct* coherent features at the two arc radii.

---

## 5. Exercises

1. **Recover β and compare to $\Lambda$CDM prediction.** Extract the mass posterior samples from `samples.csv`. For each sample, compute ray-tracing for both source planes and derive β. Plot the posterior on β and compare to the truth value (0.59 for the assumed cosmology).

2. **How much does marginalising over $z_{S1}$, $z_{S2}$ inflate the β posterior?** In reality you'd have photo-z uncertainties. Add $z_{S1}, z_{S2}$ as free parameters with Gaussian priors at the truth ± 10%. Compare the β posterior with and without marginalisation.

3. **Compare to published DSPL cosmology measurements** (Collett & Auger 2014, Smith+2021).

---

## References

- Collett & Auger (2014), MNRAS 443, 969 — DSPL cosmography.
- Gavazzi et al. (2008), ApJ 677, 1046 — first DSPL ("Jackpot" J0946+1006).
- Narayan & Bartelmann (1997) §2.3 — lens equation for multiple source planes.
- `autolens_workspace_latest/scripts/imaging/features/double_source_plane/` — canonical examples.

---

*Learning to Autolens — Examples / double_source_plane / 01*  
*Rodrigo Córdova Rosado, Harvard CfA*